# Análise Exploratória das Variáveis SIM para o Modelo Dimensional

**Objetivo:** Identificar os valores distintos de cada variável do SIM que será usada nas dimensões do star schema, verificando:
- Valores únicos (categorias reais)
- Valores ausentes (nulos/vazios)
- Compatibilidade entre anos (2014-2023)
- Distribuição de frequência

**Variáveis-alvo por dimensão:**
| Dimensão | Coluna(s) SIM |
|---|---|
| Dim_Tempo | DTOBITO |
| Dim_Municipio | CODMUNOCOR, CODMUNRES |
| Dim_FaixaEtaria | IDADE |
| Dim_RacaCor | RACACOR |
| Dim_LocalOcorrencia | LOCOCOR |
| Dim_SituacaoGestacionalObito | TPMORTEOCO |
| Dim_TipoParto | PARTO |
| Dim_MomentoObitoParto | OBITOPARTO |
| Dim_TipoGravidez | GRAVIDEZ |
| Dim_SemanaGestacao | SEMAGESTAC |
| Dim_CID | CAUSABAS, CAUSAMAT, CAUSABAS_O |
| Dim_EstabelecimentoSaude | CODESTAB |
| Atributo direto na fato | SEXO, ASSISTMED |

In [1]:
import pandas as pd
import os
import glob

print("pandas version:", pd.__version__)

pandas version: 3.0.3


In [2]:
# Configuração: pasta dos dados SIM
pasta_sim = "../arquivos/SIM"

# Lista todos os CSVs anuais, ordenados
arquivos = sorted(glob.glob(os.path.join(pasta_sim, "dados_*.csv")))
print(f"Arquivos encontrados: {len(arquivos)}")
for a in arquivos:
    print(f"  {os.path.basename(a)}")

Arquivos encontrados: 10
  dados_2014.csv
  dados_2015.csv
  dados_2016.csv
  dados_2017.csv
  dados_2018.csv
  dados_2019.csv
  dados_2020.csv
  dados_2021.csv
  dados_2022.csv
  dados_2023.csv


In [3]:
# Colunas de interesse para o modelo dimensional
COLUNAS_INTERESSE = [
    'DTOBITO',       # Dim_Tempo
    'SEXO',          # Atributo direto na fato (normalizado)
    'IDADE',         # Dim_FaixaEtaria
    'RACACOR',       # Dim_RacaCor
    'LOCOCOR',       # Dim_LocalOcorrencia
    'TPMORTEOCO',    # Dim_SituacaoGestacionalObito
    'PARTO',         # Dim_TipoParto
    'OBITOPARTO',    # Dim_MomentoObitoParto
    'GRAVIDEZ',      # Dim_TipoGravidez
    'SEMAGESTAC',    # Dim_SemanaGestacao
    'CAUSABAS',      # Dim_CID (causa básica)
    'CAUSAMAT',      # Dim_CID (causa materna)
    'CAUSABAS_O',    # Dim_CID (causa básica original)
    'CODMUNOCOR',    # Dim_Municipio (ocorrência)
    'CODMUNRES',     # Dim_Municipio (residência)
    'CODESTAB',      # Dim_EstabelecimentoSaude
    'ASSISTMED',     # recebeu_assistencia_medica
]

In [ ]:
# Função para ler e consolidar todos os anos com apenas as colunas de interesse
def carregar_dados_sim(arquivos, colunas):
    """Lê CSVs do SIM, normaliza colunas para maiúsculo e retorna DataFrame consolidado."""
    frames = []
    for arquivo in arquivos:
        ano = os.path.basename(arquivo).replace('dados_', '').replace('.csv', '')
        print(f"Lendo {os.path.basename(arquivo)}...", end=' ')
        df_ano = pd.read_csv(arquivo, sep=';', encoding='latin1', dtype=str, low_memory=False)
        df_ano.columns = df_ano.columns.str.upper()
        
        # Verifica quais colunas de interesse existem neste arquivo
        cols_presentes = [c for c in colunas if c in df_ano.columns]
        cols_ausentes = [c for c in colunas if c not in df_ano.columns]
        
        if cols_ausentes:
            print(f"(faltam: {cols_ausentes})", end=' ')
        
        df_ano = df_ano[cols_presentes].copy()
        df_ano['_ano'] = ano
        frames.append(df_ano)
        print(f"{len(df_ano)} registros")
    
    df = pd.concat(frames, ignore_index=True)
    print(f"\nTotal consolidado: {len(df)} registros ({len(frames)} anos)")
    return df

df_sim = carregar_dados_sim(arquivos, COLUNAS_INTERESSE)

Lendo dados_2014.csv... 1227039 registros
Lendo dados_2015.csv... 1264175 registros
Lendo dados_2016.csv... 1309774 registros
Lendo dados_2017.csv... 1312663 registros
Lendo dados_2018.csv... 1316719 registros
Lendo dados_2019.csv... 1349801 registros
Lendo dados_2020.csv... 1556824 registros
Lendo dados_2021.csv... 1832649 registros
Lendo dados_2022.csv... 

## 1. SEXO — Valores Originais no SIM

Mapeamento definido no plano:
| Original (SIM) | Normalizado |
|---|---|
| 'F', '2' | 'F' |
| 'M', '1' | 'M' |
| 'I', '0', '9', outros | 'I' |

Vamos verificar os valores reais presentes.

In [ ]:
# --- SEXO ---
print("=" * 60)
print("SEXO - Valores originais no SIM")
print("=" * 60)

sexo_counts = df_sim['SEXO'].value_counts(dropna=False).sort_index()
print(sexo_counts.to_string())
print(f"\nTotal: {sexo_counts.sum()}")
print(f"Nulos: {df_sim['SEXO'].isna().sum()}")

# Aplicar a normalização proposta
def normaliza_sexo(valor):
    if pd.isna(valor):
        return 'I'
    valor = str(valor).strip().upper()
    if valor in ('F', '2'):
        return 'F'
    elif valor in ('M', '1'):
        return 'M'
    else:
        return 'I'

df_sim['sexo_normalizado'] = df_sim['SEXO'].apply(normaliza_sexo)
print("\n--- Distribuição após normalização ---")
print(df_sim['sexo_normalizado'].value_counts().to_string())

In [ ]:
# Verificar consistência do SEXO por ano
print("SEXO por ano (valores originais):\n")
print(pd.crosstab(df_sim['_ano'], df_sim['SEXO'], margins=True).to_string())

## 2. IDADE — Faixa Etária

O campo IDADE no SIM tem 3 dígitos:
- 1º dígito: unidade de medida (`4` = anos)
- 2º e 3º dígitos: valor

Faixas previstas: 10-14, 15-19, 20-24, 25-29, 30-34, 35-39, 40-44, 45-49

O filtro de mortalidade materna usa `IDADE BETWEEN '410' AND '449'` (10 a 49 anos).

In [ ]:
# --- IDADE ---
print("=" * 60)
print("IDADE - Valores originais no SIM")
print("=" * 60)

# Mostrar distribuição completa
idade_counts = df_sim['IDADE'].value_counts(dropna=False).sort_index()
print(f"Total de valores distintos: {len(idade_counts)}")
print(f"\nPrimeiros 30 valores:")
print(idade_counts.head(30).to_string())
print(f"\nÚltimos 30 valores:")
print(idade_counts.tail(30).to_string())

# Verificar os primeiros dígitos (unidade de medida)
print("\n\n--- Primeiro dígito do IDADE (unidade de medida) ---")
primeiro_digito = df_sim['IDADE'].dropna().str[0].value_counts().sort_index()
print(primeiro_digito.to_string())

In [ ]:
# IDADE: analisar apenas registros com 1º dígito = 4 (medida em anos)
# e verificar a distribuição de idade (2 últimos dígitos)
df_idade_anos = df_sim[df_sim['IDADE'].str[0] == '4'].copy()
df_idade_anos['idade_valor'] = df_idade_anos['IDADE'].str[1:3].astype(int)

print("Distribuição de idade (anos) - apenas registros com unidade=4:\n")
print(df_idade_anos['idade_valor'].value_counts().sort_index().to_string())

# Faixas etárias conforme plano
faixas = [
    (10, 14, '10-14'),
    (15, 19, '15-19'),
    (20, 24, '20-24'),
    (25, 29, '25-29'),
    (30, 34, '30-34'),
    (35, 39, '35-39'),
    (40, 44, '40-44'),
    (45, 49, '45-49'),
    (50, 200, '50+'),
]

def classificar_faixa(idade):
    if pd.isna(idade):
        return None
    for min_id, max_id, nome in faixas:
        if min_id <= idade <= max_id:
            return nome
    return 'DESCONHECIDA'

df_idade_anos['faixa_etaria'] = df_idade_anos['idade_valor'].apply(classificar_faixa)
print("\n--- Distribuição por faixa etária ---")
print(df_idade_anos['faixa_etaria'].value_counts().to_string())

## 3. RACACOR — Raça/Cor

| Código | Descrição |
|---|---|
| 1 | Branca |
| 2 | Preta |
| 3 | Amarela |
| 4 | Parda |
| 5 | Indígena |

In [ ]:
# --- RACACOR ---
print("=" * 60)
print("RACACOR - Raça/Cor")
print("=" * 60)

racacor_counts = df_sim['RACACOR'].value_counts(dropna=False).sort_index()
print(racacor_counts.to_string())
print(f"\nNulos: {df_sim['RACACOR'].isna().sum()}")
print(f"Valores distintos: {racacor_counts.index.tolist()}")

## 4. LOCOCOR — Local de Ocorrência

| Código | Descrição |
|---|---|
| 1 | Hospital |
| 2 | Outros est. saúde |
| 3 | Domicílio |
| 4 | Via pública |
| 5 | Outros |
| 6 | Aldeia indígena |
| 9 | Ignorado |

In [ ]:
# --- LOCOCOR ---
print("=" * 60)
print("LOCOCOR - Local de Ocorrência")
print("=" * 60)

lococor_counts = df_sim['LOCOCOR'].value_counts(dropna=False).sort_index()
print(lococor_counts.to_string())
print(f"\nNulos: {df_sim['LOCOCOR'].isna().sum()}")
print(f"Valores distintos: {sorted(lococor_counts.index[~pd.isna(lococor_counts.index)].astype(str).tolist())}")

## 5. TPMORTEOCO — Situação Gestacional do Óbito

| Código | Descrição |
|---|---|
| 1 | Na gravidez |
| 2 | No parto |
| 3 | No abortamento |
| 4 | Até 42 dias pós-parto |
| 5 | 43d a 1 ano pós-parto |
| 8 | Não ocorreu nestes períodos |
| 9 | Ignorado |

In [ ]:
# --- TPMORTEOCO ---
print("=" * 60)
print("TPMORTEOCO - Situação Gestacional do Óbito")
print("=" * 60)

tpmorteoco_counts = df_sim['TPMORTEOCO'].value_counts(dropna=False).sort_index()
print(tpmorteoco_counts.to_string())
print(f"\nNulos: {df_sim['TPMORTEOCO'].isna().sum()}")
print(f"Valores distintos: {sorted(tpmorteoco_counts.index[~pd.isna(tpmorteoco_counts.index)].astype(str).tolist())}")

## 6. PARTO — Tipo de Parto

| Código | Descrição |
|---|---|
| 1 | Vaginal |
| 2 | Cesáreo |
| 9 | Ignorado |

In [ ]:
# --- PARTO ---
print("=" * 60)
print("PARTO - Tipo de Parto")
print("=" * 60)

parto_counts = df_sim['PARTO'].value_counts(dropna=False).sort_index()
print(parto_counts.to_string())
print(f"\nNulos: {df_sim['PARTO'].isna().sum()}")
print(f"Valores distintos: {sorted(parto_counts.index[~pd.isna(parto_counts.index)].astype(str).tolist())}")

## 7. OBITOPARTO — Momento do Óbito em relação ao Parto

| Código | Descrição |
|---|---|
| 1 | Antes |
| 2 | Durante |
| 3 | Depois |
| 9 | Ignorado |

In [ ]:
# --- OBITOPARTO ---
print("=" * 60)
print("OBITOPARTO - Momento do Óbito em relação ao Parto")
print("=" * 60)

obitoparto_counts = df_sim['OBITOPARTO'].value_counts(dropna=False).sort_index()
print(obitoparto_counts.to_string())
print(f"\nNulos: {df_sim['OBITOPARTO'].isna().sum()}")
print(f"Valores distintos: {sorted(obitoparto_counts.index[~pd.isna(obitoparto_counts.index)].astype(str).tolist())}")

## 8. GRAVIDEZ — Tipo de Gravidez

| Código | Descrição |
|---|---|
| 1 | Única |
| 2 | Dupla |
| 3 | Tripla e mais |
| 9 | Ignorada |

In [ ]:
# --- GRAVIDEZ ---
print("=" * 60)
print("GRAVIDEZ - Tipo de Gravidez")
print("=" * 60)

gravidez_counts = df_sim['GRAVIDEZ'].value_counts(dropna=False).sort_index()
print(gravidez_counts.to_string())
print(f"\nNulos: {df_sim['GRAVIDEZ'].isna().sum()}")
print(f"Valores distintos: {sorted(gravidez_counts.index[~pd.isna(gravidez_counts.index)].astype(str).tolist())}")

## 9. SEMAGESTAC — Semana de Gestação

| Faixa | Semanas |
|---|---|
| 0-21 | 0 a 21 |
| 22-27 | 22 a 27 |
| 28-31 | 28 a 31 |
| 32-36 | 32 a 36 |
| 37-41 | 37 a 41 |
| 42+ | 42 ou mais |

In [ ]:
# --- SEMAGESTAC ---
print("=" * 60)
print("SEMAGESTAC - Semana de Gestação")
print("=" * 60)

semagestac_counts = df_sim['SEMAGESTAC'].value_counts(dropna=False).sort_index()
print(semagestac_counts.to_string())
print(f"\nNulos: {df_sim['SEMAGESTAC'].isna().sum()}")
print(f"Valores distintos: {sorted(semagestac_counts.index[~pd.isna(semagestac_counts.index)].astype(str).tolist())}")

## 10. CID — Causas (Role-Playing Dimension)

Três colunas de CID que apontam para a mesma Dim_CID:
- **CAUSABAS**: Causa básica (NOT NULL na fato)
- **CAUSAMAT**: Causa materna
- **CAUSABAS_O**: Causa básica original

In [ ]:
# --- CAUSABAS (causa básica) ---
print("=" * 60)
print("CAUSABAS - Causa Básica (NOT NULL)")
print("=" * 60)

causabas_counts = df_sim['CAUSABAS'].value_counts(dropna=False).sort_index()
print(f"Total de CIDs distintos: {len(causabas_counts)}")
print(f"Nulos: {df_sim['CAUSABAS'].isna().sum()}")
print(f"\nTop 20 CIDs mais frequentes:")
print(causabas_counts.head(20).to_string())
print(f"\nBottom 10 CIDs menos frequentes:")
print(causabas_counts.tail(10).to_string())

In [ ]:
# --- CAUSAMAT (causa materna) ---
print("=" * 60)
print("CAUSAMAT - Causa Materna")
print("=" * 60)

causamat_counts = df_sim['CAUSAMAT'].value_counts(dropna=False).sort_index()
print(f"Total de CIDs distintos: {len(causamat_counts)}")
print(f"Nulos: {df_sim['CAUSAMAT'].isna().sum()}")
# Muitos nulos são esperados, pois nem todo óbito tem causa materna
nao_nulos = df_sim['CAUSAMAT'].notna().sum()
print(f"Não nulos: {nao_nulos} ({nao_nulos/len(df_sim)*100:.1f}%)")
print(f"\nTop 20 CIDs maternos mais frequentes:")
print(causamat_counts.head(20).to_string())

In [ ]:
# --- CAUSABAS_O (causa básica original) ---
print("=" * 60)
print("CAUSABAS_O - Causa Básica Original")
print("=" * 60)

causabas_o_counts = df_sim['CAUSABAS_O'].value_counts(dropna=False).sort_index()
print(f"Total de CIDs distintos: {len(causabas_o_counts)}")
print(f"Nulos: {df_sim['CAUSABAS_O'].isna().sum()}")
nao_nulos = df_sim['CAUSABAS_O'].notna().sum()
print(f"Não nulos: {nao_nulos} ({nao_nulos/len(df_sim)*100:.1f}%)")
print(f"\nTop 20 CIDs originais mais frequentes:")
print(causabas_o_counts.head(20).to_string())

## 11. DTOBITO — Data do Óbito (Dim_Tempo)

Verificar amplitude de datas, anos cobertos e valores malformados.

In [ ]:
# --- DTOBITO ---
print("=" * 60)
print("DTOBITO - Data do Óbito")
print("=" * 60)

# Primeiro, verificar formatos das datas
print("Amostra de 20 valores de DTOBITO:")
print(df_sim['DTOBITO'].dropna().sample(20, random_state=42).to_string())

# Extrair ano
df_sim['ano_dtobito'] = df_sim['DTOBITO'].str[:4]
print(f"\n--- Distribuição de anos ---")
ano_counts = df_sim['ano_dtobito'].value_counts(dropna=False).sort_index()
print(ano_counts.to_string())

# Verificar valores com formato inesperado
# Assumindo formato YYYYMMDD ou similar
print(f"\n--- Valores com comprimento diferente de 8 ---")
tamanhos = df_sim['DTOBITO'].dropna().str.len().value_counts().sort_index()
print(f"Distribuição de comprimentos: {tamanhos.to_dict()}")

# Verificar nulos
print(f"\nNulos em DTOBITO: {df_sim['DTOBITO'].isna().sum()}")

## 12. CODMUNOCOR e CODMUNRES — Municípios

A mesma Dim_Municipio é usada como role-playing (ocorrência e residência).

In [ ]:
# --- CODMUNOCOR (Município de Ocorrência) ---
print("=" * 60)
print("CODMUNOCOR - Município de Ocorrência")
print("=" * 60)

mun_ocor_counts = df_sim['CODMUNOCOR'].value_counts(dropna=False).sort_index()
print(f"Total de municípios distintos: {len(mun_ocor_counts)}")
print(f"Nulos: {df_sim['CODMUNOCOR'].isna().sum()}")
print(f"\nTop 10 municípios:")
print(mun_ocor_counts.head(10).to_string())
print(f"\nÚltimos 5:")
print(mun_ocor_counts.tail(5).to_string())

# Verificar comprimento do código
print(f"\n--- Comprimento do código ---")
print(df_sim['CODMUNOCOR'].dropna().str.len().value_counts().sort_index().to_string())

In [ ]:
# --- CODMUNRES (Município de Residência) ---
print("=" * 60)
print("CODMUNRES - Município de Residência")
print("=" * 60)

mun_res_counts = df_sim['CODMUNRES'].value_counts(dropna=False).sort_index()
print(f"Total de municípios distintos: {len(mun_res_counts)}")
print(f"Nulos: {df_sim['CODMUNRES'].isna().sum()}")
print(f"\nTop 10 municípios:")
print(mun_res_counts.head(10).to_string())

# Comparar cobertura
print(f"\n--- Comparação Ocorrência vs Residência ---")
print(f"Municípios apenas em CODMUNOCOR: {len(set(mun_ocor_counts.index) - set(mun_res_counts.index))}")
print(f"Municípios apenas em CODMUNRES: {len(set(mun_res_counts.index) - set(mun_ocor_counts.index))}")
print(f"Municípios em ambos: {len(set(mun_ocor_counts.index) & set(mun_res_counts.index))}")

## 13. CODESTAB — Estabelecimento de Saúde (CNES)

Código do estabelecimento para vincular com Dim_EstabelecimentoSaude.

In [ ]:
# --- CODESTAB ---
print("=" * 60)
print("CODESTAB - Estabelecimento de Saúde")
print("=" * 60)

codestab_counts = df_sim['CODESTAB'].value_counts(dropna=False).sort_index()
print(f"Total de estabelecimentos distintos: {len(codestab_counts)}")
print(f"Nulos: {df_sim['CODESTAB'].isna().sum()}")
nao_nulos = df_sim['CODESTAB'].notna().sum()
print(f"Não nulos: {nao_nulos} ({nao_nulos/len(df_sim)*100:.1f}%)")
print(f"\nTop 10 estabelecimentos:")
print(codestab_counts.head(10).to_string())
print(f"\nComprimento do código:")
print(df_sim['CODESTAB'].dropna().str.len().value_counts().sort_index().to_string())

## 14. ASSISTMED — Assistência Médica

| Código | Descrição |
|---|---|
| 1 | Sim |
| 2 (ou ausente) | Não / Ignorado |

In [ ]:
# --- ASSISTMED ---
print("=" * 60)
print("ASSISTMED - Recebeu Assistência Médica")
print("=" * 60)

assistmed_counts = df_sim['ASSISTMED'].value_counts(dropna=False).sort_index()
print(assistmed_counts.to_string())
print(f"\nNulos: {df_sim['ASSISTMED'].isna().sum()}")
print(f"Valores distintos: {sorted(assistmed_counts.index[~pd.isna(assistmed_counts.index)].astype(str).tolist())}")

## 15. APLICAÇÃO DO FILTRO DE MORTALIDADE MATERNA

Validar o filtro definido no plano:

```sql
WHERE (SEXO = 'F' AND IDADE BETWEEN '410' AND '449')
   OR (SEXO IN ('M', 'I') AND TPMORTEOCO IN ('1', '2', '3', '4', '5'))
```

In [ ]:
# --- APLICAR FILTRO DE MORTALIDADE MATERNA ---
print("=" * 60)
print("FILTRO DE MORTALIDADE MATERNA")
print("=" * 60)

# Aplicar o filtro usando sexo_normalizado e IDADE como string
# e TPMORTEOCO como string (dados lidos como str)
mask_feminino_fertil = (
    (df_sim['sexo_normalizado'] == 'F') &
    (df_sim['IDADE'].between('410', '449'))
)

mask_excecao_mi = (
    (df_sim['sexo_normalizado'].isin(['M', 'I'])) &
    (df_sim['TPMORTEOCO'].isin(['1', '2', '3', '4', '5']))
)

total_geral = len(df_sim)
total_filtro = mask_feminino_fertil.sum() + mask_excecao_mi.sum()

print(f"Total de registros no dataset completo: {total_geral}")
print(f"")
print(f"--- Filtro principal (F, 10-49 anos): {mask_feminino_fertil.sum():,} ({mask_feminino_fertil.sum()/total_geral*100:.1f}%)")
print(f"--- Exceção (M/I com TPMORTEOCO): {mask_excecao_mi.sum():,} ({mask_excecao_mi.sum()/total_geral*100:.1f}%)")
print(f"")
print(f"Total de registros elegíveis para a fato: {total_filtro:,} ({total_filtro/total_geral*100:.1f}%)")

# Detalhar a exceção
if mask_excecao_mi.sum() > 0:
    print(f"\n--- Detalhamento da exceção M/I ---")
    excecao = df_sim[mask_excecao_mi]
    print(f"Por SEXO normalizado:")
    print(excecao['sexo_normalizado'].value_counts().to_string())
    print(f"\nPor TPMORTEOCO:")
    print(excecao['TPMORTEOCO'].value_counts().to_string())

## 16. RESUMO — Valores Distintos por Variável

Tabela consolidada para validação do DDL.

In [ ]:
# Resumo consolidado
print("=" * 80)
print("RESUMO - Valores Distintos por Variável para o Modelo Dimensional")
print("=" * 80)

variaveis = [
    ('SEXO', 'Atributo direto', df_sim['SEXO'].dropna().unique().tolist()),
    ('IDADE (1º dígito)', 'Dim_FaixaEtaria', df_sim['IDADE'].dropna().str[0].unique().tolist()),
    ('RACACOR', 'Dim_RacaCor', sorted(df_sim['RACACOR'].dropna().unique())),
    ('LOCOCOR', 'Dim_LocalOcorrencia', sorted(df_sim['LOCOCOR'].dropna().unique())),
    ('TPMORTEOCO', 'Dim_SituacaoGestacional', sorted(df_sim['TPMORTEOCO'].dropna().unique())),
    ('PARTO', 'Dim_TipoParto', sorted(df_sim['PARTO'].dropna().unique())),
    ('OBITOPARTO', 'Dim_MomentoObitoParto', sorted(df_sim['OBITOPARTO'].dropna().unique())),
    ('GRAVIDEZ', 'Dim_TipoGravidez', sorted(df_sim['GRAVIDEZ'].dropna().unique())),
    ('SEMAGESTAC', 'Dim_SemanaGestacao', sorted(df_sim['SEMAGESTAC'].dropna().unique())),
    ('ASSISTMED', 'Atributo direto', sorted(df_sim['ASSISTMED'].dropna().unique())),
]

for var, dim, valores in variaveis:
    nulos = df_sim[var].isna().sum() if var in df_sim.columns else 0
    print(f"\n{var:20s} → {dim:25s} | Nulos: {nulos:>8,} | Distintos: {len(valores):>4} | {valores}")

print(f"\n\n--- CID ---")
for col in ['CAUSABAS', 'CAUSAMAT', 'CAUSABAS_O']:
    n_cids = df_sim[col].nunique(dropna=False)
    nulos = df_sim[col].isna().sum()
    print(f"{col:20s} → Dim_CID (role-playing) | Nulos: {nulos:>8,} | CIDs distintos: {n_cids:>6,}")

print(f"\n--- Municípios ---")
for col in ['CODMUNOCOR', 'CODMUNRES']:
    n_muns = df_sim[col].nunique(dropna=False)
    nulos = df_sim[col].isna().sum()
    print(f"{col:20s} → Dim_Municipio (role-playing) | Nulos: {nulos:>8,} | Municípios distintos: {n_muns:>6,}")

print(f"\n--- Estabelecimentos ---")
n_estab = df_sim['CODESTAB'].nunique(dropna=False)
nulos_estab = df_sim['CODESTAB'].isna().sum()
print(f"{'CODESTAB':20s} → Dim_EstabelecimentoSaude       | Nulos: {nulos_estab:>8,} | Estabelecimentos distintos: {n_estab:>6,}")

print(f"\n--- Datas ---")
print(f"{'DTOBITO':20s} → Dim_Tempo                       | Nulos: {df_sim['DTOBITO'].isna().sum():>8,} | Datas distintas: {df_sim['DTOBITO'].nunique():>6,} | Anos: {sorted(df_sim['ano_dtobito'].dropna().unique())}")

In [ ]:
# Consolidado final: verificar compatibilidade dos valores com o DDL
print("=" * 80)
print("VALIDAÇÃO DE COMPATIBILIDADE COM O DDL")
print("=" * 80)

# Regras de validação
validacoes = [
    ("SEXO", "Deve conter apenas {'F','M','I', nulo}", 
     set(df_sim['SEXO'].dropna().unique()) <= {'F', 'M', 'I', '1', '2', '0', '9'}),
    ("RACACOR", "Deve conter apenas {'1','2','3','4','5', nulo}",
     set(df_sim['RACACOR'].dropna().unique()) <= {'1', '2', '3', '4', '5'}),
    ("LOCOCOR", "Deve conter apenas {'1','2','3','4','5','6','9', nulo}",
     set(df_sim['LOCOCOR'].dropna().unique()) <= {'1', '2', '3', '4', '5', '6', '9'}),
    ("TPMORTEOCO", "Deve conter apenas {'1','2','3','4','5','8','9', nulo}",
     set(df_sim['TPMORTEOCO'].dropna().unique()) <= {'1', '2', '3', '4', '5', '8', '9'}),
    ("PARTO", "Deve conter apenas {'1','2','9', nulo}",
     set(df_sim['PARTO'].dropna().unique()) <= {'1', '2', '9'}),
    ("OBITOPARTO", "Deve conter apenas {'1','2','3','9', nulo}",
     set(df_sim['OBITOPARTO'].dropna().unique()) <= {'1', '2', '3', '9'}),
    ("GRAVIDEZ", "Deve conter apenas {'1','2','3','9', nulo}",
     set(df_sim['GRAVIDEZ'].dropna().unique()) <= {'1', '2', '3', '9'}),
    ("CAUSABAS", "Não deve conter nulos (NOT NULL na fato)",
     df_sim['CAUSABAS'].notna().all()),
    ("CODMUNOCOR", "Não deve conter nulos (NOT NULL na fato)",
     df_sim['CODMUNOCOR'].notna().all()),
]

print(f"{'Variável':20s} {'Esperado':<45s} {'Status':<10s}")
print("-" * 75)
for var, esperado, resultado in validacoes:
    status = "✓ OK" if resultado else "✗ PROBLEMA"
    print(f"{var:20s} {esperado:<45s} {status:<10s}")